# Notebook 1 - Cleaning & Extraction Pipeline
### Rancang Bangun Chatbot Sumber Informasi Sivitas UPI Berbasis RAG

**Focus:** ingestion, PDF/HTML extraction, OCR fallback, preprocessing,
Markdown generation for RAG.

| Stage | What it does | Output |
|-------|--------------|--------|
| 1 | Recursive scan of dataset roots + optional polite web crawl | `manifest.json` |
| 2 | PDF text extraction (PyMuPDF) -> OCR fallback (PaddleOCR/Tesseract); HTML extraction (trafilatura) | `raw/*.json` |
| 3 | Cleaning + Markdown generation | `cleaned/*.json`, `*.txt`, `*.md` |
| 3b | Markdown quality validation | `logs/markdown_validation.json` |

**Outputs** (all under `Dataset/_pipeline/`): `manifest.json`, `raw/`, `cleaned/`,
`cleaned/cleaned_manifest.json`, `logs/`.

**Key properties:** recursive ingestion, resumable, atomic writes, graceful error
handling. Generates `.md` as the PRIMARY source for downstream chunking + RAG.


## Cell 1 - Install dependencies

In [ ]:
# =============================================================================
# Notebook 1 - Install dependencies
# =============================================================================
# Supports BOTH Google Colab AND local Windows execution.
# PaddleOCR = primary OCR engine; Tesseract = automatic fallback.

import sys, subprocess

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _pip_install(*pkgs):
    """Run pip install for the given packages, quiet but non-silent on errors."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *pkgs]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError as e:
        print(f"  pip install failed for {pkgs}: {e}")


if _in_colab():
    get_ipython().system('pip install -q pymupdf==1.24.10 trafilatura==1.12.2 beautifulsoup4==4.12.3 lxml==5.3.0 requests==2.32.3 pdf2image==1.17.0 pytesseract==0.3.13 tqdm==4.66.5 pandas==2.2.2')
    get_ipython().system('pip install -q paddlepaddle==2.6.2 paddleocr==2.8.1 2>/dev/null || echo "PaddleOCR install reported issues - notebook will fall back to Tesseract."')
    get_ipython().system('apt-get -qq update')
    get_ipython().system('apt-get -qq install -y tesseract-ocr tesseract-ocr-ind tesseract-ocr-eng poppler-utils')
    print("\nColab install step complete.")
else:
    # --- Local execution (Windows / macOS / Linux) ---
    # Auto-install the Python deps the notebook needs. Heavy ones (paddlepaddle/
    # paddleocr) included so a clean machine works without a separate pip step.
    print(f"Local environment detected ({sys.platform}). Installing Python deps...")
    _pip_install(
        "pymupdf", "trafilatura", "beautifulsoup4", "lxml", "requests",
        "pdf2image", "pytesseract", "tqdm", "pandas",
        "paddlepaddle", "paddleocr",
    )
    # Report what is detected.
    for mod in ("fitz", "trafilatura", "bs4", "lxml", "tqdm",
                "paddleocr", "pytesseract"):
        try:
            __import__(mod); print(f"  [ok]   {mod}")
        except Exception as e:
            print(f"  [miss] {mod}  ({e.__class__.__name__})")

## Cell 2 - Imports, environment detection, logging

*Always run this and Cell 3 first in a new session.*

In [ ]:
# =============================================================================
# Notebook 1 - Imports, environment detection, logging
# =============================================================================
import json
import re
import os
import logging
import hashlib
import time
import io
import traceback
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin, urlparse

from tqdm.auto import tqdm

# --- Environment: try Colab first, otherwise stay local (Windows-friendly) ---
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Local mode (no Colab). Using local Windows-safe paths in Cell 3.")


# --- Logging: timestamped, idempotent, also writes to a file later ---
def get_logger(name="nb1", logfile=None):
    """Configured logger. Safe to call repeatedly (no duplicate handlers)."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s",
                            datefmt="%H:%M:%S")
    if not any(isinstance(h, logging.StreamHandler) for h in logger.handlers):
        sh = logging.StreamHandler()
        sh.setFormatter(fmt)
        logger.addHandler(sh)
    if logfile and not any(isinstance(h, logging.FileHandler) for h in logger.handlers):
        try:
            fh = logging.FileHandler(logfile, encoding="utf-8")
            fh.setFormatter(fmt)
            logger.addHandler(fh)
        except Exception:
            pass
    logger.propagate = False
    return logger


log = get_logger()
log.info("Imports ready. In Colab: %s", IN_COLAB)


## Cell 3 - Configuration

Append a folder to `CONFIG["dataset_roots"]` to add a new source. Nothing else is hardcoded.

In [ ]:
# =============================================================================
# Notebook 1 - Configuration  (edit here only)
# =============================================================================
# Append a folder to CONFIG["dataset_roots"] to add a new source.
# Set CONFIG["web_sources"] to enable polite crawling. Nothing else is hardcoded.

# Base path resolution order:
#   1. RAG_UPI_BASE env var (overrides everything)
#   2. Colab Drive mount  (/content/drive/MyDrive/Dataset)
#   3. Local Windows project root (D:\Project\RAG_UPI\Dataset)
_env_base = os.environ.get("RAG_UPI_BASE")
if _env_base:
    DRIVE_BASE = Path(_env_base)
elif IN_COLAB:
    DRIVE_BASE = Path("/content/drive/MyDrive/Dataset")
else:
    DRIVE_BASE = Path(r"D:\Project\RAG_UPI\Dataset")

CONFIG = {
    "dataset_roots": [
        DRIVE_BASE,
        DRIVE_BASE / "PPID",
        DRIVE_BASE / "LPPM_UPI",
        DRIVE_BASE / "Dataset_PMB_UPI",
    ],

    "web_sources": [
        # "https://dit-pendidikan.upi.edu/",
    ],
    "crawl_max_pages": 40,
    "crawl_delay_seconds": 2.0,
    "crawl_same_domain_only": True,
    "crawl_max_retries": 2,

    "pipeline_dir": DRIVE_BASE / "_pipeline",

    "doc_extensions": [".pdf", ".html", ".htm", ".txt"],

    "ocr_languages_paddle": "id",
    "ocr_languages_tesseract": "ind+eng",
    "ocr_min_chars_per_page": 80,
    "ocr_dpi": 150,

    # --- Markdown generation settings (NEW) ---
    "markdown_min_chars": 120,
    "markdown_heading_size_factor": 1.15,
    "markdown_max_heading_level": 3,
    "markdown_front_matter": True,
}

PIPE = CONFIG["pipeline_dir"]
DIRS = {
    "downloads": PIPE / "downloads",
    "raw":       PIPE / "raw",
    "cleaned":   PIPE / "cleaned",   # JSON + TXT + MD all land here
    "logs":      PIPE / "logs",
}
MANIFEST_PATH = PIPE / "manifest.json"

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

log = get_logger("nb1", logfile=str(DIRS["logs"] / "nb1_extraction.log"))


def doc_id(source):
    """Stable 16-char id derived from a document's source path/URL."""
    return hashlib.sha1(str(source).encode("utf-8")).hexdigest()[:16]


def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as e:
        log.warning("Corrupt JSON %s (%s) - returning default.", path, e)
        return default


def write_json(path, obj):
    """Atomic JSON write (write to .tmp then replace)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


def write_text_atomic(path, text):
    """Atomic UTF-8 text write (Windows-safe)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)


def category_of(source_path):
    """Infer a category label from which dataset root a file lives under."""
    s = str(source_path)
    best = "Dataset_root"
    for root in CONFIG["dataset_roots"]:
        if root == DRIVE_BASE:
            continue
        if s.startswith(str(root)):
            best = root.name
    return best


log.info("Config loaded. Pipeline dir: %s", PIPE)
log.info("Base dir: %s", DRIVE_BASE)
log.info("Dataset roots: %d | Web sources: %d",
         len(CONFIG["dataset_roots"]), len(CONFIG["web_sources"]))


## Cell 4 - Stage 1: Ingestion

Recursively scans all dataset roots. Optional polite web crawl when `web_sources` is set.

In [ ]:
# =============================================================================
# Notebook 1 - Stage 1: Ingestion (recursive scan + optional polite web crawl)
# =============================================================================

def scan_local_documents():
    """Recursively collect supported documents from every dataset root.

    Returns a list of manifest rows. De-duplicates by absolute path so the
    same file under overlapping roots is not ingested twice.
    """
    exts = set(CONFIG["doc_extensions"])
    seen, items = set(), []
    for root in CONFIG["dataset_roots"]:
        root = Path(root)
        if not root.exists():
            log.warning("Dataset root missing, skipped: %s", root)
            continue
        for path in root.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in exts:
                continue
            if str(path).startswith(str(CONFIG["pipeline_dir"])):
                continue  # never ingest our own outputs
            key = str(path.resolve())
            if key in seen:
                continue
            seen.add(key)
            items.append({
                "id": doc_id(key),
                "source": str(path),
                "origin": "local",
                "title": path.stem,
                "type": path.suffix.lower().lstrip("."),
                "category": category_of(path),
                "status": "discovered",
            })
    return items


# --- Anti-bot detection: we SKIP protected pages, never bypass them ---
BLOCK_SIGNS = ("cloudflare", "captcha", "access denied", "verify you are human",
               "attention required", "checking your browser")


def _looks_blocked(resp):
    """Heuristic check for an anti-bot wall. True -> skip and log (no bypass)."""
    if resp.status_code in (401, 403, 429, 503):
        return True
    body = (resp.text or "")[:5000].lower()
    return any(sign in body for sign in BLOCK_SIGNS)


def _polite_get(session, url):
    """GET with polite retries on transient failures. Returns response or None."""
    for attempt in range(1, CONFIG["crawl_max_retries"] + 2):
        try:
            resp = session.get(url, timeout=25)
            if resp.status_code in (429, 503) and attempt <= CONFIG["crawl_max_retries"]:
                wait = CONFIG["crawl_delay_seconds"] * (attempt + 1)
                log.info("Transient %s on %s - waiting %.1fs then retrying.",
                         resp.status_code, url, wait)
                time.sleep(wait)
                continue
            return resp
        except Exception as e:
            if attempt <= CONFIG["crawl_max_retries"]:
                time.sleep(CONFIG["crawl_delay_seconds"] * attempt)
                continue
            log.warning("Request failed after retries: %s (%s)", url, e)
            return None
    return None


def crawl_web_sources():
    """Breadth-first crawl of public UPI pages to discover documents + articles.

    Legitimate access only. Pages protected by Cloudflare / CAPTCHA / rate
    limiting are politely retried once, then skipped and logged. This function
    does NOT bypass protection, solve CAPTCHAs, or use stealth evasion.

    HTML article pages are themselves recorded as ingestable items (origin=web,
    type=html); linked PDFs are downloaded to DIRS["downloads"].
    """
    seeds = CONFIG["web_sources"]
    if not seeds:
        log.info("No web sources configured - skipping crawl.")
        return []

    import requests
    from bs4 import BeautifulSoup

    session = requests.Session()
    session.headers.update({
        "User-Agent": "UPI-Thesis-RAG/1.0 (academic research; contact: thesis project)"
    })

    queue = list(seeds)
    visited, found = set(), []
    pdf_exts = {".pdf"}

    pbar = tqdm(total=CONFIG["crawl_max_pages"], desc="Crawling web", unit="page")
    while queue and len(visited) < CONFIG["crawl_max_pages"]:
        url = queue.pop(0).split("#")[0]
        if url in visited:
            continue
        visited.add(url)
        pbar.update(1)

        resp = _polite_get(session, url)
        if resp is None:
            found.append(_web_row(url, "page", "request_failed"))
            continue
        if _looks_blocked(resp):
            log.warning("Protected page skipped (no bypass attempted): %s", url)
            found.append(_web_row(url, "page", "blocked_skipped"))
            continue
        ctype = resp.headers.get("Content-Type", "")
        if resp.status_code != 200 or "text/html" not in ctype:
            continue

        # The HTML page itself is an ingestable article.
        page_path = DIRS["downloads"] / f"{doc_id(url)}.html"
        try:
            page_path.write_text(resp.text, encoding="utf-8")
            row = _web_row(url, "html", "discovered")
            row["source"] = str(page_path)   # local copy for extraction
            row["url"] = url
            found.append(row)
        except Exception as e:
            log.warning("Could not save HTML for %s (%s)", url, e)

        # Discover links: PDFs are downloaded; same-domain pages are queued.
        soup = BeautifulSoup(resp.text, "lxml")
        base_domain = urlparse(url).netloc
        for a in soup.find_all("a", href=True):
            link = urljoin(url, a["href"]).split("#")[0]
            suffix = Path(urlparse(link).path).suffix.lower()
            if suffix in pdf_exts:
                row = _download_pdf(session, link, a.get_text(strip=True))
                if row:
                    found.append(row)
            elif link.startswith("http") and link not in visited:
                if not CONFIG["crawl_same_domain_only"] or urlparse(link).netloc == base_domain:
                    queue.append(link)
        time.sleep(CONFIG["crawl_delay_seconds"])
    pbar.close()
    return found


def _web_row(url, dtype, status):
    return {"id": doc_id(url), "source": url, "origin": "web",
            "title": url, "type": dtype, "category": "web", "status": status}


def _download_pdf(session, url, title):
    """Download a public PDF link to DIRS['downloads']. Skips if already present."""
    out = DIRS["downloads"] / f"{doc_id(url)}.pdf"
    if out.exists():
        row = _web_row(url, "pdf", "discovered")
        row["source"], row["url"], row["title"] = str(out), url, (title or out.stem)
        return row
    resp = _polite_get(session, url)
    if resp is None or resp.status_code != 200:
        log.warning("PDF download skipped: %s", url)
        return _web_row(url, "pdf", "download_failed")
    try:
        out.write_bytes(resp.content)
    except Exception as e:
        log.warning("Could not save PDF %s (%s)", url, e)
        return _web_row(url, "pdf", "download_failed")
    row = _web_row(url, "pdf", "discovered")
    row["source"], row["url"], row["title"] = str(out), url, (title or out.stem)
    return row


def build_manifest(do_crawl=None):
    """Run ingestion; merge into a persistent manifest without duplicating rows."""
    if do_crawl is None:
        do_crawl = bool(CONFIG["web_sources"])
    log.info("Stage 1: scanning %d local roots...", len(CONFIG["dataset_roots"]))
    items = scan_local_documents()
    if do_crawl:
        items += crawl_web_sources()

    existing = {row["id"]: row for row in read_json(MANIFEST_PATH, default=[])}
    for it in items:
        if it["id"] in existing:
            # Preserve processing status; refresh descriptive fields only.
            for k in ("title", "type", "category", "source"):
                if k in it:
                    existing[it["id"]][k] = it[k]
        else:
            existing[it["id"]] = it
    manifest = list(existing.values())
    write_json(MANIFEST_PATH, manifest)

    by_status, by_type = {}, {}
    for row in manifest:
        by_status[row["status"]] = by_status.get(row["status"], 0) + 1
        by_type[row["type"]] = by_type.get(row["type"], 0) + 1
    log.info("Manifest saved: %s (%d items)", MANIFEST_PATH, len(manifest))
    log.info("By status: %s", by_status)
    log.info("By type: %s", by_type)
    return manifest


# ---- RUN STAGE 1 ----
manifest = build_manifest()

## Cell 5 - Stage 2: Extraction (PDF text + OCR fallback + HTML)

PyMuPDF for text + layout spans; PaddleOCR primary, Tesseract fallback. Auto-locates `tesseract.exe` from common Windows install paths.

In [ ]:
# === Workaround: PaddleOCR 3.x oneDNN/PIR bug on Windows + force Tesseract ===
# Paste this as a new cell BEFORE Cell 5 (the extraction cell), then re-run from Cell 2.

import os

# 1) If you ever want to retry PaddleOCR, these env vars dodge the
#    "ConvertPirAttribute2RuntimeAttribute not support" crash:
os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_enable_pir_in_executor"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"

# 2) Force Tesseract for this run (fast & proven). Skip Paddle entirely
#    by making the import fail cleanly inside _init_ocr().
import sys, types
if "paddleocr" in sys.modules:
    del sys.modules["paddleocr"]
# Block future imports of paddleocr in this session:
_blocked = types.ModuleType("paddleocr")
def _raise(*a, **k):
    raise ImportError("paddleocr disabled for this run; using Tesseract")
_blocked.PaddleOCR = _raise
sys.modules["paddleocr"] = _blocked

# 3) Reset OCR engine state so _init_ocr() runs fresh
_OCR_ENGINE = "uninitialised"
_paddle_ocr = None
print("Tesseract forced. Paddle disabled for this session.")

In [ ]:
import json
removed = 0
for f in DIRS["raw"].glob("*.json"):
    d = json.loads(f.read_text(encoding="utf-8"))
    bad = sum(1 for p in d.get("pages", [])
              if p.get("method") == "ocr" and p.get("n_chars", 0) < 10)
    if bad and bad / max(len(d.get("pages", [])), 1) >= 0.5:
        f.unlink(); removed += 1

m = read_json(MANIFEST_PATH, default=[])
for row in m:
    if row.get("status") in ("empty_extraction", "extract_error"):
        row["status"] = "discovered"
write_json(MANIFEST_PATH, m)
print(f"Cleared {removed} bad raws; reset failed statuses.")

In [ ]:
# =============================================================================
# Notebook 1 - Stage 2: Extraction (PDF text -> OCR fallback; HTML extraction)
# =============================================================================
# Captures per-page text PLUS lightweight layout hints (font sizes, bold flags)
# so the Markdown stage can infer headings without re-opening the PDF.
import fitz  # PyMuPDF
import shutil

_paddle_ocr = None
_OCR_ENGINE = "uninitialised"


def _locate_tesseract():
    """Find tesseract.exe even when PATH wasn't refreshed in the Jupyter server.

    Order: PATH -> TESSERACT_CMD env var -> common Windows install locations.
    Sets pytesseract.pytesseract.tesseract_cmd when found.
    """
    try:
        import pytesseract
    except Exception:
        return None
    # 1) PATH
    found = shutil.which("tesseract")
    if found:
        pytesseract.pytesseract.tesseract_cmd = found
        return found
    # 2) Env var override
    env_cmd = os.environ.get("TESSERACT_CMD")
    if env_cmd and Path(env_cmd).exists():
        pytesseract.pytesseract.tesseract_cmd = env_cmd
        return env_cmd
    # 3) Common Windows install locations
    candidates = [
        r"C:\Program Files\Tesseract-OCR\tesseract.exe",
        r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
        os.path.expandvars(r"%LOCALAPPDATA%\Programs\Tesseract-OCR\tesseract.exe"),
        os.path.expandvars(r"%USERPROFILE%\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"),
    ]
    for c in candidates:
        if c and Path(c).exists():
            pytesseract.pytesseract.tesseract_cmd = c
            # Also add its folder to PATH so its data files resolve.
            os.environ["PATH"] = str(Path(c).parent) + os.pathsep + os.environ.get("PATH", "")
            return c
    return None


def _init_ocr():
    """Initialise OCR. Tries PaddleOCR 3.x signature first, then 2.x, then Tesseract."""
    global _paddle_ocr, _OCR_ENGINE
    if _OCR_ENGINE != "uninitialised":
        return _OCR_ENGINE
    lang = CONFIG["ocr_languages_paddle"]
    try:
        from paddleocr import PaddleOCR
        # Try in order: new 3.x signature, old 2.x signature, minimal signature.
        attempts = [
            dict(use_textline_orientation=True, lang=lang),
            dict(use_angle_cls=True, lang=lang, show_log=False),
            dict(use_angle_cls=True, lang=lang),
            dict(lang=lang),
        ]
        last_err = None
        for kwargs in attempts:
            try:
                _paddle_ocr = PaddleOCR(**kwargs)
                _OCR_ENGINE = "paddle"
                log.info("OCR engine: PaddleOCR (lang=%s, signature=%s).",
                         lang, list(kwargs.keys()))
                return _OCR_ENGINE
            except Exception as e:
                last_err = e
                continue
        raise last_err if last_err else RuntimeError("PaddleOCR init failed")
    except Exception as e:
        log.warning("PaddleOCR unavailable (%s) - falling back to Tesseract.", e)
        try:
            import pytesseract  # noqa: F401
            tcmd = _locate_tesseract()
            if not tcmd:
                _OCR_ENGINE = "none"
                log.error("Tesseract binary not found. Install UB-Mannheim "
                          "Tesseract or set TESSERACT_CMD env var to the .exe path. "
                          "Scanned pages will be empty.")
                return _OCR_ENGINE
            _OCR_ENGINE = "tesseract"
            log.info("OCR engine: Tesseract at %s (lang=%s).",
                     tcmd, CONFIG["ocr_languages_tesseract"])
        except Exception as e2:
            _OCR_ENGINE = "none"
            log.error("No OCR engine available (%s). Scanned pages will be empty.", e2)
    return _OCR_ENGINE


def _paddle_extract_lines(result):
    """Pull text lines out of any known PaddleOCR result shape.

    Handles both 2.x (list-of-list-of-[box,(text,conf)]) and 3.x
    (list of dicts with 'rec_texts' / OCRResult objects).
    """
    lines = []
    if not result:
        return lines
    for page in result:
        # 3.x: dict-like with 'rec_texts'
        if isinstance(page, dict) and "rec_texts" in page:
            lines.extend(t for t in page["rec_texts"] if t)
            continue
        # 3.x OCRResult object
        if hasattr(page, "json") and not isinstance(page, (list, tuple)):
            try:
                data = page.json
                if isinstance(data, dict) and "rec_texts" in data:
                    lines.extend(t for t in data["rec_texts"] if t)
                    continue
            except Exception:
                pass
        # 2.x: list of [box, (text, conf)]
        if isinstance(page, (list, tuple)):
            for entry in page:
                if entry and len(entry) >= 2 and entry[1]:
                    txt = entry[1][0] if isinstance(entry[1], (list, tuple)) else entry[1]
                    if txt:
                        lines.append(txt)
    return lines


def _ocr_image(pil_image):
    """Run OCR on a PIL image, returning extracted text. Engine-agnostic."""
    engine = _init_ocr()
    if engine == "paddle":
        import numpy as np
        arr = np.array(pil_image)
        # 3.x prefers .predict(); 2.x uses .ocr(img, cls=True). Try both.
        result = None
        if hasattr(_paddle_ocr, "predict"):
            try:
                result = _paddle_ocr.predict(arr)
            except Exception:
                result = None
        if result is None:
            try:
                result = _paddle_ocr.ocr(arr, cls=True)
            except TypeError:
                result = _paddle_ocr.ocr(arr)
        return "\n".join(_paddle_extract_lines(result))
    if engine == "tesseract":
        import pytesseract
        return pytesseract.image_to_string(
            pil_image, lang=CONFIG["ocr_languages_tesseract"])
    return ""


def _extract_pdf_layout(page):
    """Return (plain_text, spans) for one PyMuPDF page.

    spans = list of {text, size, bold, y} in reading order, used by the
    Markdown generator to infer heading levels.
    """
    plain = page.get_text("text").strip()
    spans = []
    try:
        d = page.get_text("dict")
        for block in d.get("blocks", []):
            if block.get("type", 0) != 0:
                continue
            for line in block.get("lines", []):
                line_text = "".join(s.get("text", "") for s in line.get("spans", []))
                line_text = line_text.strip()
                if not line_text:
                    continue
                sizes = [s.get("size", 0) for s in line.get("spans", [])]
                flags = [s.get("flags", 0) for s in line.get("spans", [])]
                bold = any(f & 16 for f in flags)
                size = max(sizes) if sizes else 0
                y0 = block.get("bbox", [0, 0, 0, 0])[1]
                spans.append({"text": line_text, "size": round(size, 2),
                              "bold": bool(bold), "y": round(y0, 1)})
    except Exception:
        pass
    return plain, spans


def extract_pdf(path):
    """Extract a PDF page-by-page; OCR fallback for low-text pages."""
    from PIL import Image
    pages = []
    doc = fitz.open(path)
    try:
        for i, page in enumerate(doc):
            text, spans = _extract_pdf_layout(page)
            method = "text"
            if len(text) < CONFIG["ocr_min_chars_per_page"]:
                try:
                    pix = page.get_pixmap(dpi=CONFIG["ocr_dpi"])
                    img = Image.open(io.BytesIO(pix.tobytes("png")))
                    ocr_text = _ocr_image(img).strip()
                    if len(ocr_text) > len(text):
                        text, method, spans = ocr_text, "ocr", []
                except Exception as e:
                    log.warning("OCR failed %s p%d: %s", Path(path).name, i + 1, e)
            pages.append({"page": i + 1, "text": text, "method": method,
                          "n_chars": len(text), "spans": spans})
    finally:
        doc.close()
    return pages


def extract_html(path_or_text, is_text=False):
    """Extract main article content from HTML."""
    raw = path_or_text if is_text else Path(path_or_text).read_text(
        encoding="utf-8", errors="ignore")

    text = ""
    try:
        import trafilatura
        text = trafilatura.extract(
            raw, include_comments=False, include_tables=True,
            favor_recall=True) or ""
    except Exception as e:
        log.warning("trafilatura failed (%s) - using BeautifulSoup fallback.", e)

    if len(text.strip()) < 100:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(raw, "lxml")
        for tag in soup(["nav", "header", "footer", "script", "style",
                         "aside", "form", "noscript"]):
            tag.decompose()
        main = soup.find("main") or soup.find("article") or soup.body or soup
        text = main.get_text("\n") if main else ""

    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return [{"page": 1, "text": text, "method": "html",
             "n_chars": len(text), "spans": []}]


def run_extraction():
    """Stage 2: extract every ingested document. Skips items already extracted."""
    _init_ocr()
    manifest = read_json(MANIFEST_PATH, default=[])
    if not manifest:
        log.error("No manifest found. Run Stage 1 first.")
        return

    todo = [r for r in manifest
            if r["status"] in ("discovered",)
            and r["type"] in ("pdf", "html", "htm", "txt")
            and not (DIRS["raw"] / f"{r['id']}.json").exists()]
    log.info("Stage 2: %d documents to extract (%d already on disk).",
             len(todo), sum(1 for r in manifest
                            if (DIRS["raw"] / f"{r['id']}.json").exists()))

    status = {r["id"]: r for r in manifest}
    for row in tqdm(todo, desc="Extracting", unit="doc"):
        src = Path(row["source"])
        try:
            if not src.exists():
                status[row["id"]]["status"] = "missing_file"
                continue
            ext = src.suffix.lower()
            if ext == ".pdf":
                pages = extract_pdf(src)
                source_type = "pdf"
            elif ext in (".html", ".htm"):
                pages = extract_html(src)
                source_type = "html"
            elif ext == ".txt":
                t = src.read_text(encoding="utf-8", errors="ignore").strip()
                pages = [{"page": 1, "text": t, "method": "text",
                          "n_chars": len(t), "spans": []}]
                source_type = "txt"
            else:
                status[row["id"]]["status"] = "unsupported_format"
                continue

            total_chars = sum(p["n_chars"] for p in pages)
            if total_chars == 0:
                status[row["id"]]["status"] = "empty_extraction"
                log.warning("Empty extraction: %s", src.name)
                continue

            write_json(DIRS["raw"] / f"{row['id']}.json", {
                "id": row["id"], "source": row["source"],
                "url": row.get("url"), "title": row["title"],
                "category": row["category"], "source_type": source_type,
                "n_pages": len(pages),
                "ocr_pages": sum(1 for p in pages if p["method"] == "ocr"),
                "total_chars": total_chars, "pages": pages,
                "extracted_at": datetime.now().isoformat(timespec="seconds"),
            })
            status[row["id"]]["status"] = "extracted"
        except Exception as e:
            log.warning("Extraction failed %s: %s", src.name, e)
            log.debug(traceback.format_exc())
            status[row["id"]]["status"] = "extract_error"

    write_json(MANIFEST_PATH, list(status.values()))
    log.info("Stage 2 complete. OCR engine used: %s. Raw JSON in: %s",
             _OCR_ENGINE, DIRS["raw"])


# ---- RUN STAGE 2 ----
run_extraction()


## Cell 6 - Stage 3: Cleaning + Markdown generation

Per document, writes `<id>.json` + `<id>.txt` + **`<id>.md`** under `cleaned/`. The `.md` is the primary input for chunking.

In [ ]:
# =============================================================================
# Notebook 1 - Stage 3: Cleaning + Markdown generation
# =============================================================================
# Cleaning preserves semantic structure (headings, lists, paragraphs).
# Output per document (all under DIRS["cleaned"]):
#     <id>.json   - structured per-page data + metadata
#     <id>.txt    - flat cleaned text
#     <id>.md     - RAG-friendly Markdown (PRIMARY source for chunking)

PAGE_NUM_RE   = re.compile(r"^\s*(?:hal(?:aman)?\.?\s*)?[-]?\s*\d{1,4}\s*(?:/\s*\d{1,4})?\s*[-]?\s*$", re.I)
BULLET_RE     = re.compile(r"^\s*([\-\*o]|\d+[\.\)]|[a-z][\.\)])\s+", re.I)
NUM_LIST_RE   = re.compile(r"^\s*(\d+)[\.\)]\s+(.*)$")
MULTISPACE_RE = re.compile(r"[ \t]{2,}")
OCR_NOISE_RE  = re.compile(r"(.)\1{4,}")
HEADING_RE    = re.compile(r"^([A-Z0-9][A-Z0-9 .,:'\-/()]{3,80})$")
URL_RE        = re.compile(r"https?://\S+")
BOILERPLATE = {
    "beranda", "kontak", "login", "logout", "menu", "search", "cari",
    "copyright", "hak cipta dilindungi", "all rights reserved",
    "universitas pendidikan indonesia", "bagikan", "share", "read more",
    "selengkapnya", "previous", "next", "sebelumnya", "selanjutnya",
}


def _detect_repeated_lines(pages, min_ratio=0.4):
    from collections import Counter
    counts, n = Counter(), max(len(pages), 1)
    for p in pages:
        for ln in {x.strip() for x in p["text"].splitlines() if x.strip()}:
            if 3 <= len(ln) <= 90:
                counts[ln] += 1
    return {ln for ln, c in counts.items() if n > 2 and c / n >= min_ratio}


def _is_heading(line):
    s = line.strip()
    return bool(HEADING_RE.match(s)) and len(s.split()) <= 12


def _is_boilerplate(line):
    return line.strip().lower() in BOILERPLATE


def clean_text(text, repeated):
    """Clean one page; preserve headings (##), bullets (-), numbered lists."""
    out = []
    for raw_line in text.splitlines():
        s = raw_line.strip()
        if not s:
            out.append("")
            continue
        if s in repeated or PAGE_NUM_RE.match(s) or _is_boilerplate(s):
            continue
        s = OCR_NOISE_RE.sub(r"\1", s)
        s = URL_RE.sub("", s).strip()
        s = MULTISPACE_RE.sub(" ", s)
        if not s:
            continue
        m = NUM_LIST_RE.match(s)
        if m:
            out.append(f"{m.group(1)}. {m.group(2).strip()}")
        elif BULLET_RE.match(s):
            out.append("- " + BULLET_RE.sub("", s).strip())
        elif _is_heading(s):
            out.append("\n## " + s + "\n")
        else:
            out.append(s)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()


# ----------------------- Markdown generation -------------------------------

def _infer_heading_levels_from_spans(pages, factor, max_level):
    """Return dict {heading_text_lower: level (1..max_level)}.

    Uses PDF font-size spans (when present). Empty if no span info (OCR-only).
    """
    import statistics
    sizes = []
    for p in pages:
        for s in p.get("spans", []) or []:
            if s.get("size"):
                sizes.append(s["size"])
    if not sizes:
        return {}
    body = statistics.median(sizes)
    threshold = body * factor
    big = sorted({sp["size"] for p in pages for sp in (p.get("spans") or [])
                  if sp["size"] >= threshold}, reverse=True)
    size_to_level = {}
    for idx, sz in enumerate(big):
        size_to_level[sz] = min(idx + 1, max_level)

    headings = {}
    for p in pages:
        for sp in p.get("spans", []) or []:
            if sp["size"] in size_to_level:
                key = sp["text"].strip().lower()
                if 3 <= len(key) <= 120:
                    headings[key] = size_to_level[sp["size"]]
    return headings


def _front_matter(doc):
    """Build a YAML front-matter block for RAG ingestion."""
    fm = {
        "id": doc["id"],
        "title": doc["title"],
        "category": doc["category"],
        "source_type": doc["source_type"],
        "n_pages": doc["n_pages"],
        "source": doc["source"],
    }
    if doc.get("url"):
        fm["url"] = doc["url"]
    lines = ["---"]
    for k, v in fm.items():
        v_str = str(v).replace('"', "'")
        lines.append(f'{k}: "{v_str}"')
    lines.append("---\n")
    return "\n".join(lines)


def _dedupe_consecutive_lines(text):
    """Drop a line if identical to the previous non-blank line."""
    out, prev = [], None
    for ln in text.splitlines():
        s = ln.rstrip()
        if s and s == prev:
            continue
        out.append(ln)
        if s:
            prev = s
    return "\n".join(out)


def build_markdown(doc, clean_pages):
    """Produce RAG-friendly Markdown for one document.

    1. Use heading hints inferred from PDF spans to upgrade '## ' lines.
    2. Preserve bullet/numbered lists and paragraphs from clean_text().
    3. Emit '<!-- page: N -->' separators for multi-page PDFs.
    4. Optional YAML front-matter for downstream metadata access.
    """
    heading_map = _infer_heading_levels_from_spans(
        doc.get("pages", []),
        CONFIG["markdown_heading_size_factor"],
        CONFIG["markdown_max_heading_level"],
    )

    md_parts = []
    title = (doc.get("title") or "").strip()
    if title:
        md_parts.append(f"# {title}\n")

    for p in clean_pages:
        body = p["text"]
        if not body:
            continue
        lines = []
        for ln in body.splitlines():
            stripped = ln.strip()
            if not stripped:
                lines.append("")
                continue
            if stripped.startswith("## "):
                key = stripped[3:].strip().lower()
                lvl = heading_map.get(key)
                if lvl:
                    lines.append(("#" * lvl) + " " + stripped[3:].strip())
                else:
                    lines.append(stripped)
            else:
                lvl = heading_map.get(stripped.lower())
                if lvl and 3 <= len(stripped) <= 120:
                    lines.append(("#" * lvl) + " " + stripped)
                else:
                    lines.append(ln)
        page_md = "\n".join(lines).strip()
        if doc.get("source_type") == "pdf" and doc.get("n_pages", 1) > 1:
            md_parts.append(f"<!-- page: {p['page']} -->")
        md_parts.append(page_md)

    md = "\n\n".join(part for part in md_parts if part).strip() + "\n"
    md = _dedupe_consecutive_lines(md)
    md = re.sub(r"\n{3,}", "\n\n", md)

    if CONFIG["markdown_front_matter"]:
        md = _front_matter(doc) + md
    return md


# --------------------------- Stage 3 driver --------------------------------
def run_cleaning():
    """Stage 3: clean every extracted document, emit JSON+TXT+MD.

    Re-runs when any of the three outputs is missing - so old runs that lacked
    .md files get the .md added on the next run.
    """
    raw_files = sorted(DIRS["raw"].glob("*.json"))
    if not raw_files:
        log.error("No extracted files found. Run Stage 2 first.")
        return

    def _outputs_complete(stem):
        return all((DIRS["cleaned"] / f"{stem}{ext}").exists()
                   for ext in (".json", ".txt", ".md"))

    todo = [f for f in raw_files if not _outputs_complete(f.stem)]
    log.info("Stage 3: %d documents to (re)clean (%d already complete).",
             len(todo), len(raw_files) - len(todo))

    cleaned_manifest = []
    for f in tqdm(todo, desc="Cleaning + MD", unit="doc"):
        doc = read_json(f)
        if not doc:
            continue
        repeated = _detect_repeated_lines(doc["pages"])
        clean_pages, kept_chars = [], 0
        for p in doc["pages"]:
            ct = clean_text(p["text"], repeated)
            kept_chars += len(ct)
            clean_pages.append({"page": p["page"], "text": ct,
                                "method": p.get("method", "text")})
        full_text = "\n\n".join(p["text"] for p in clean_pages if p["text"])

        md_text = build_markdown(doc, clean_pages)
        md_path = DIRS["cleaned"] / f"{doc['id']}.md"
        txt_path = DIRS["cleaned"] / f"{doc['id']}.txt"
        json_path = DIRS["cleaned"] / f"{doc['id']}.json"

        out = {
            "id": doc["id"], "source": doc["source"], "url": doc.get("url"),
            "title": doc["title"], "category": doc["category"],
            "source_type": doc["source_type"], "n_pages": doc["n_pages"],
            "raw_chars": doc["total_chars"],
            "clean_chars": kept_chars,
            "markdown_chars": len(md_text),
            "markdown_file": md_path.name,
            "pages": clean_pages,
            "cleaned_at": datetime.now().isoformat(timespec="seconds"),
        }
        write_json(json_path, out)
        write_text_atomic(txt_path, full_text)
        write_text_atomic(md_path, md_text)

        cleaned_manifest.append({
            "id": doc["id"], "title": doc["title"], "category": doc["category"],
            "source_type": doc["source_type"], "n_pages": doc["n_pages"],
            "clean_chars": kept_chars, "markdown_chars": len(md_text),
            "markdown_file": md_path.name,
        })

    cm_path = DIRS["cleaned"] / "cleaned_manifest.json"
    existing = {r["id"]: r for r in read_json(cm_path, default=[])}
    for r in cleaned_manifest:
        existing[r["id"]] = r
    write_json(cm_path, list(existing.values()))
    log.info("Stage 3 complete. .json/.txt/.md in: %s", DIRS["cleaned"])
    log.info("Cleaned manifest: %s (%d docs)", cm_path, len(existing))


# ---- RUN STAGE 3 ----
run_cleaning()


## Cell 6b - Markdown Quality Validation

Validates every `.md` for: empty content, OCR noise residue, fragmented tokens, duplicate sections, header/footer leakage. Writes `logs/markdown_validation.json` and WARN log lines.

In [ ]:
# =============================================================================
# Notebook 1 - Markdown Quality Validation
# =============================================================================
# Reads every .md under DIRS["cleaned"] and flags:
#   - empty markdown        (< CONFIG["markdown_min_chars"] real content)
#   - OCR failure residue   (long runs of non-letter chars, fragmented words)
#   - duplicate sections    (same heading appears more than N times)
#   - header/footer leakage (a short line repeats across page separators)
# Produces logs/markdown_validation.json plus per-issue WARN log lines.

_HEADING_LINE = re.compile(r"^(#{1,6})\s+(.+?)\s*$")
_NON_LETTER_RUN = re.compile(r"[^\w\s]{8,}")


def _strip_front_matter(md):
    if md.startswith("---\n"):
        end = md.find("\n---", 4)
        if end != -1:
            return md[end + 4:].lstrip()
    return md


def validate_markdown(md_path):
    """Return a dict of validation findings for one .md file."""
    md_path = Path(md_path)
    raw = md_path.read_text(encoding="utf-8", errors="ignore")
    body = _strip_front_matter(raw)
    issues = []

    real_chars = len(re.sub(r"\s+", "", body))
    if real_chars < CONFIG["markdown_min_chars"]:
        issues.append({"type": "empty_markdown",
                       "detail": f"only {real_chars} non-whitespace chars"})

    if _NON_LETTER_RUN.search(body):
        issues.append({"type": "ocr_noise_residue",
                       "detail": "long non-letter run detected"})
    tokens = re.findall(r"\w+", body)
    if tokens:
        short = sum(1 for t in tokens if len(t) <= 2)
        if short / len(tokens) > 0.45 and len(tokens) > 200:
            issues.append({"type": "fragmented_text",
                           "detail": f"{short}/{len(tokens)} tokens <=2 chars"})

    headings = []
    for ln in body.splitlines():
        m = _HEADING_LINE.match(ln.strip())
        if m:
            headings.append(m.group(2).strip().lower())
    from collections import Counter
    dup = [(h, c) for h, c in Counter(headings).items() if c >= 4]
    for h, c in dup:
        issues.append({"type": "duplicate_section",
                       "detail": f"'{h[:60]}' x{c}"})

    page_chunks = re.split(r"<!--\s*page:\s*\d+\s*-->", body)
    if len(page_chunks) >= 4:
        firsts, lasts = [], []
        for chunk in page_chunks:
            lines = [ln.strip() for ln in chunk.splitlines() if ln.strip()]
            if lines:
                firsts.append(lines[0])
                lasts.append(lines[-1])
        for label, lines in (("header", firsts), ("footer", lasts)):
            cnt = Counter(lines)
            for ln, c in cnt.items():
                if c >= max(4, int(len(lines) * 0.5)) and 3 < len(ln) < 100:
                    issues.append({"type": f"{label}_leakage",
                                   "detail": f"'{ln[:80]}' on {c} pages"})

    return {"id": md_path.stem, "file": md_path.name,
            "chars": len(raw), "real_chars": real_chars,
            "issues": issues}


def run_validation():
    md_files = sorted(DIRS["cleaned"].glob("*.md"))
    if not md_files:
        log.warning("No .md files found - run Stage 3 first.")
        return []
    reports = []
    issue_counts = {}
    for f in tqdm(md_files, desc="Validating MD", unit="doc"):
        rep = validate_markdown(f)
        reports.append(rep)
        for it in rep["issues"]:
            issue_counts[it["type"]] = issue_counts.get(it["type"], 0) + 1
            log.warning("MD issue [%s] %s: %s",
                        it["type"], rep["id"], it["detail"])

    summary = {
        "validated_at": datetime.now().isoformat(timespec="seconds"),
        "n_docs": len(reports),
        "n_with_issues": sum(1 for r in reports if r["issues"]),
        "issue_counts": issue_counts,
        "reports": reports,
    }
    write_json(DIRS["logs"] / "markdown_validation.json", summary)
    log.info("Markdown validation: %d/%d docs clean.",
             summary["n_docs"] - summary["n_with_issues"], summary["n_docs"])
    log.info("Issue counts: %s", issue_counts)
    return summary


# ---- RUN VALIDATION ----
markdown_validation = run_validation()


In [ ]:
import re
from collections import Counter

# Patch: tighten duplicate_section detection so table cells / acronyms
# don't get flagged as duplicate headings.
_PURE_NUMBER = re.compile(r"^[\d.,\-\s]+$")
_HEADING_LINE_LOCAL = re.compile(r"^(#{1,6})\s+(.+?)\s*$")

def validate_markdown(md_path):
    md_path = Path(md_path)
    raw = md_path.read_text(encoding="utf-8", errors="ignore")
    body = _strip_front_matter(raw)
    issues = []

    real_chars = len(re.sub(r"\s+", "", body))
    if real_chars < CONFIG["markdown_min_chars"]:
        issues.append({"type": "empty_markdown",
                       "detail": f"only {real_chars} non-whitespace chars"})

    if _NON_LETTER_RUN.search(body):
        issues.append({"type": "ocr_noise_residue",
                       "detail": "long non-letter run detected"})

    tokens = re.findall(r"\w+", body)
    if tokens:
        short = sum(1 for t in tokens if len(t) <= 2)
        if short / len(tokens) > 0.45 and len(tokens) > 200:
            issues.append({"type": "fragmented_text",
                           "detail": f"{short}/{len(tokens)} tokens <=2 chars"})

    # Duplicate section: only flag if the "heading" looks like real prose
    # (not a number, not <8 chars, has a space → multi-word).
    headings = []
    for ln in body.splitlines():
        m = _HEADING_LINE_LOCAL.match(ln.strip())
        if not m:
            continue
        h = m.group(2).strip().lower()
        if len(h) < 8: continue                          # skip 'fpips' etc.
        if _PURE_NUMBER.match(h): continue               # skip '1.500.000'
        if " " not in h: continue                        # require multi-word
        headings.append(h)
    for h, c in Counter(headings).items():
        if c >= 6:                                        # was >=4
            issues.append({"type": "duplicate_section",
                           "detail": f"'{h[:60]}' x{c}"})

    # Header/footer leakage (unchanged)
    page_chunks = re.split(r"<!--\s*page:\s*\d+\s*-->", body)
    if len(page_chunks) >= 4:
        firsts, lasts = [], []
        for chunk in page_chunks:
            lines = [ln.strip() for ln in chunk.splitlines() if ln.strip()]
            if lines:
                firsts.append(lines[0])
                lasts.append(lines[-1])
        for label, lines in (("header", firsts), ("footer", lasts)):
            cnt = Counter(lines)
            for ln, c in cnt.items():
                if c >= max(4, int(len(lines) * 0.5)) and 3 < len(ln) < 100:
                    issues.append({"type": f"{label}_leakage",
                                   "detail": f"'{ln[:80]}' on {c} pages"})

    return {"id": md_path.stem, "file": md_path.name,
            "chars": len(raw), "real_chars": real_chars,
            "issues": issues}

markdown_validation = run_validation()

## Cell 7 - SELF-REVIEW: failure points, risks & mitigations

**Architectural weaknesses & mitigations applied**
- *PaddleOCR install fragility* — lazy init; automatic Tesseract fallback if
  PaddleOCR import/init fails. Notebook never hard-crashes on OCR.
- *Tesseract not on PATH after install* — Cell 5 auto-locates `tesseract.exe`
  from common Windows install dirs even when the Jupyter server inherited an
  outdated PATH. Override with the `TESSERACT_CMD` env var if needed.
- *Drive / FS I/O interrupts* — all writes go through atomic `.tmp -> replace`.
- *Re-running duplicates work* — every stage skips items already on disk;
  cleaning re-runs only if .json/.txt/.md are not all present.

**Failure points & handling**
- Missing dataset root -> warning, scan continues.
- Unreadable / encrypted / corrupt PDF -> per-doc try/except, `extract_error`.
- 0-byte PDF -> caught by PyMuPDF as "Cannot open empty file"; status set to
  `extract_error`. Delete or re-download these source files.
- Empty extraction (OCR fully failed) -> status `empty_extraction`, excluded
  from cleaning.
- Blocked web page -> `blocked_skipped`, no bypass.

**Markdown-specific risks**
- *False-positive headings*: heuristic font-size band + ALL-CAPS rule;
  validator flags duplicate sections.
- *OCR pages have no font-size info*: heading inference falls back to
  ALL-CAPS-only rule.
- *Table reconstruction*: PyMuPDF doesn't preserve tables; they surface as
  text blocks. Add pdfplumber if table fidelity matters.
- *Header/footer leakage*: validator detects lines repeating across page
  separators; cleaning already strips >=40%-page repeats.

**Dependency risks**
- PaddleOCR API drift across 2.x / 3.x — `_init_ocr` tries multiple signatures
  and `_ocr_image` tries both `.predict()` and `.ocr()`.
- trafilatura sparse on JS-heavy pages: BeautifulSoup fallback at < 100 chars.

**Runtime & memory**
- OCR dominates cost; born-digital PDFs skip it. Resumable across sessions.
- Pages processed one document at a time; pixmaps released per page.


## Cell 8 - TESTING & VERIFICATION SUITE

Sanity checks + Markdown tests. Run after Stages 1-3 and Validation.

In [ ]:
# =============================================================================
# Notebook 1 - TESTING & VERIFICATION SUITE
# =============================================================================
# Sanity checks + Markdown tests. Run AFTER Stages 1-3 and Validation.

def _check(name, condition, detail=""):
    tag = "PASS" if condition else "FAIL"
    log.info("[%s] %s %s", tag, name, f"- {detail}" if detail else "")
    return bool(condition)


def test_paths_exist():
    ok = True
    for k, d in DIRS.items():
        ok &= _check(f"dir:{k}", Path(d).exists(), str(d))
    ok &= _check("manifest.json present", MANIFEST_PATH.exists())
    return ok


def test_recursive_scan():
    items = scan_local_documents()
    ok = _check("scan returns list", isinstance(items, list))
    bad = [i for i in items
           if Path(i["source"]).suffix.lower() not in set(CONFIG["doc_extensions"])]
    ok &= _check("scan respects doc_extensions", not bad,
                 f"{len(bad)} unexpected types")
    ok &= _check("scan ids unique",
                 len({i["id"] for i in items}) == len(items))
    return ok


def test_pdf_extraction():
    raw = sorted(DIRS["raw"].glob("*.json"))
    if not raw:
        log.info("[WARN] test_pdf_extraction - no raw files yet.")
        return True
    ok = True
    for f in raw[:25]:
        doc = read_json(f)
        ok &= _check(f"{f.stem}: has pages", bool(doc and doc.get("pages")))
        if doc and doc.get("pages"):
            nums = [p["page"] for p in doc["pages"]]
            ok &= _check(f"{f.stem}: page order preserved",
                         nums == sorted(nums))
            ok &= _check(f"{f.stem}: schema keys present",
                         all(k in doc for k in
                             ("id", "source_type", "n_pages", "total_chars")))
    return ok


def test_ocr_fallback():
    raw = [read_json(f) for f in DIRS["raw"].glob("*.json")]
    raw = [d for d in raw if d]
    ocr_docs = [d for d in raw if d.get("ocr_pages", 0) > 0]
    log.info("[INFO] OCR engine = %s | %d/%d docs used OCR on >=1 page.",
             _OCR_ENGINE, len(ocr_docs), len(raw))
    return _check("OCR fallback logic consistent", True, "see INFO line above")


def test_html_extraction_quality():
    raw = [read_json(f) for f in DIRS["raw"].glob("*.json")]
    html = [d for d in raw if d and d.get("source_type") == "html"]
    if not html:
        log.info("[WARN] test_html_extraction_quality - no HTML docs.")
        return True
    weak = [d for d in html if d.get("total_chars", 0) < 200]
    return _check("HTML extraction substantive",
                  len(weak) <= len(html) * 0.3,
                  f"{len(weak)}/{len(html)} docs under 200 chars")


def test_cleaning_outputs():
    cj = [f for f in DIRS["cleaned"].glob("*.json") if f.name != "cleaned_manifest.json"]
    if not cj:
        log.info("[WARN] test_cleaning_outputs - no cleaned files yet.")
        return True
    ok = True
    for f in sorted(cj)[:50]:
        ok &= _check(f"{f.stem}: .txt exists",
                     (DIRS["cleaned"] / f"{f.stem}.txt").exists())
        ok &= _check(f"{f.stem}: .md exists",
                     (DIRS["cleaned"] / f"{f.stem}.md").exists())
        doc = read_json(f)
        if doc:
            ratio = doc["clean_chars"] / max(doc["raw_chars"], 1)
            ok &= _check(f"{f.stem}: retained >=10% of text",
                         ratio >= 0.10, f"kept {ratio:.0%}")
    return ok


def test_markdown_quality():
    """All .md files: content present, UTF-8 valid, mostly structured."""
    md_files = sorted(DIRS["cleaned"].glob("*.md"))
    if not md_files:
        log.info("[WARN] test_markdown_quality - no .md files yet.")
        return True
    ok = True
    empty = 0
    no_struct = 0
    for f in md_files[:50]:
        try:
            body = f.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            ok &= _check(f"{f.stem}: utf-8 readable", False)
            continue
        if len(body.strip()) < CONFIG["markdown_min_chars"]:
            empty += 1
        if (not re.search(r"^#{1,6}\s", body, flags=re.M)
                and not re.search(r"^- ", body, flags=re.M)
                and len(body) < 400):
            no_struct += 1
    ok &= _check("most .md files non-empty",
                 empty <= len(md_files) * 0.2,
                 f"{empty}/{len(md_files)} below {CONFIG['markdown_min_chars']} chars")
    ok &= _check("most .md files have some structure or content",
                 no_struct <= len(md_files) * 0.3,
                 f"{no_struct}/{len(md_files)} lack headings/lists and are short")
    return ok


def test_metadata_consistency():
    manifest = {r["id"]: r for r in read_json(MANIFEST_PATH, default=[])}
    cm = read_json(DIRS["cleaned"] / "cleaned_manifest.json", default=[])
    ok = True
    for r in cm:
        ok &= _check(f"{r['id']}: id in manifest", r["id"] in manifest)
        ok &= _check(f"{r['id']}: markdown_file recorded",
                     "markdown_file" in r and r["markdown_file"].endswith(".md"))
    return ok


def test_validation_report_present():
    rep = read_json(DIRS["logs"] / "markdown_validation.json")
    if rep is None:
        log.info("[WARN] markdown_validation.json missing - run validation cell.")
        return True
    return _check("validation report has reports list",
                  isinstance(rep.get("reports"), list),
                  f"{rep.get('n_docs')} docs validated")


def run_all_tests():
    log.info("=" * 60)
    log.info("NOTEBOOK 1 - TEST SUITE")
    log.info("=" * 60)
    results = {
        "paths": test_paths_exist(),
        "recursive_scan": test_recursive_scan(),
        "pdf_extraction": test_pdf_extraction(),
        "ocr_fallback": test_ocr_fallback(),
        "html_quality": test_html_extraction_quality(),
        "cleaning": test_cleaning_outputs(),
        "markdown": test_markdown_quality(),
        "validation_report": test_validation_report_present(),
        "metadata": test_metadata_consistency(),
    }
    passed = sum(1 for v in results.values() if v)
    log.info("=" * 60)
    log.info("RESULT: %d/%d test groups passed.", passed, len(results))
    log.info("Detail: %s", results)
    return results


def inspect_document(doc_index=0, source="cleaned"):
    folder = DIRS[source]
    files = sorted(p for p in folder.glob("*.json")
                   if p.name != "cleaned_manifest.json")
    if not files:
        print(f"No files in {folder}")
        return
    doc = read_json(files[doc_index % len(files)])
    print(f"--- {source.upper()} doc {doc_index} ---")
    print(f"title         : {doc.get('title')}")
    print(f"category      : {doc.get('category')}")
    print(f"source_type   : {doc.get('source_type')}")
    print(f"n_pages       : {doc.get('n_pages')}")
    print(f"markdown_file : {doc.get('markdown_file')}")
    print(f"markdown_chars: {doc.get('markdown_chars')}")
    for p in doc.get("pages", [])[:2]:
        print(f"\n[page {p['page']} | method={p.get('method')}]")
        print(p["text"][:600])


def preview_markdown(doc_index=0, n_chars=1200):
    """Print the first N chars of one generated .md file."""
    files = sorted(DIRS["cleaned"].glob("*.md"))
    if not files:
        print("No .md files yet.")
        return
    f = files[doc_index % len(files)]
    print(f"--- {f.name} ---")
    print(f.read_text(encoding="utf-8")[:n_chars])


test_results = run_all_tests()
print("\nTip: inspect_document(0) or preview_markdown(0) to view outputs.")


---
### Re-running individual stages (Notebook 1)
Run Cell 2 + Cell 3 first, then call any stage:
`build_manifest()` | `run_extraction()` | `run_cleaning()` |
`run_validation()` | `run_all_tests()`.

Inspect with `inspect_document(0)` or `preview_markdown(0)`.

**Outputs per document** (under `Dataset/_pipeline/cleaned/`):
- `<id>.json` — structured metadata + per-page cleaned text
- `<id>.txt`  — flat cleaned text
- `<id>.md`   — RAG-friendly Markdown (PRIMARY source for `02_chunking_pipeline.ipynb`)

**Next:** open `02_chunking_pipeline.ipynb`.
